In [1]:
# [COLAB SETUP]
import sys
import os

if "google.colab" in sys.modules:
    print("Running in Google Colab. Setting up environment...")
    
    # Mount Google Drive to persist the datasets and cloned repository
    from google.colab import drive
    drive.mount('/content/drive')
    
    repo_path = '/content/drive/MyDrive/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit'
    
    if not os.path.exists(repo_path):
        print(f"Cloning repository into {repo_path}...")
        os.makedirs('/content/drive/MyDrive', exist_ok=True)
        os.system(f'git clone https://github.com/Maleesha-K/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit.git {repo_path}')
        
    os.chdir(repo_path + '/data_pipeline')
    print("Installing base dependencies...")
    os.system('pip install -q pandas scikit-learn gdown')
    print("Setup complete!")


In [2]:
output_dir = 'datasets/finetuning'
preprocessed_dir = 'datasets/preprocessed'


In [3]:
import os
import subprocess
import pandas as pd
import json

# Auto-resolve the project root if running manually
if not os.path.exists("Makefile") and os.path.exists("../../Makefile"):
    os.chdir("../../")

os.makedirs(output_dir, exist_ok=True)

TRAIN_GDRIVE_LINK = "https://drive.google.com/uc?id=1RNioMTZEmS0g5FR8gWoYQtKgj3dvdmEi"
VAL_GDRIVE_LINK = "https://drive.google.com/uc?id=1NIjeKZvZwfIHr2XcHSj358PesUomUog5"

try:
    import gdown
except ImportError:
    print("Installing gdown...")
    subprocess.run(["pip", "install", "-q", "gdown"])
    import gdown

print(f"Downloading training dataset...")
train_csv_path = os.path.join(output_dir, "train.csv")
gdown.download(url=TRAIN_GDRIVE_LINK, output=train_csv_path)

print(f"\nDownloading validation dataset...")
val_csv_path = os.path.join(output_dir, "val.csv")
gdown.download(url=VAL_GDRIVE_LINK, output=val_csv_path)

# Unified Label Mapping
LABEL_MAP = {
    "sinhala": "sin",
    "sanskrit": "san",
    "pali": "pli"
}

def transform_to_jsonl(csv_path, jsonl_path):
    df = pd.read_csv(csv_path)
    if 'label' in df.columns:
        df['label'] = df['label'].str.lower().map(lambda x: LABEL_MAP.get(x, x))
    
    # Select columns as in benchmark transformation (id, text, label, source)
    cols_to_keep = ['id', 'text', 'label', 'source']
    # keep only existing columns
    cols_to_keep = [c for c in cols_to_keep if c in df.columns]
    
    df = df[cols_to_keep]
    
    with open(jsonl_path, 'w', encoding='utf-8') as f:
        for record in df.to_dict(orient='records'):
            f.write(json.dumps(record, ensure_ascii=False) + '\n')
    
    print(f"Transformed {csv_path} -> {jsonl_path} ({len(df)} records)")
    return df

print("\nTransforming datasets to JSONL...")
train_df = transform_to_jsonl(train_csv_path, os.path.join(output_dir, "train.jsonl"))
val_df = transform_to_jsonl(val_csv_path, os.path.join(output_dir, "val.jsonl"))


Downloading...
From: https://drive.google.com/uc?id=1RNioMTZEmS0g5FR8gWoYQtKgj3dvdmEi
To: d:\Projects\ML Projects\LangID - DSE project\data_pipeline\datasets\finetuning\train.csv
100%|██████████| 70.0M/70.0M [00:23<00:00, 2.94MB/s]


Downloading...
From: https://drive.google.com/uc?id=1NIjeKZvZwfIHr2XcHSj358PesUomUog5
To: d:\Projects\ML Projects\LangID - DSE project\data_pipeline\datasets\finetuning\val.csv
100%|██████████| 8.24M/8.24M [00:02<00:00, 3.45MB/s]



Transforming datasets to JSONL...
Transformed datasets/finetuning\train.csv -> datasets/finetuning\train.jsonl (60285 records)
Transformed datasets/finetuning\val.csv -> datasets/finetuning\val.jsonl (6986 records)


In [4]:
import random

print("\nCreating mixed validation set (val_mixed.jsonl) to track Catastrophic Forgetting...")

# Target languages we are benchmarking, excluding the new finetuning languages
OLD_LANGUAGES = ["eng", "tam", "hin", "ben", "arb", "fra", "deu"]

# Sample a maximum of N sentences per old language from flores_plus
MAX_SAMPLES_PER_OLD_LANG = 500
old_lang_samples = []

flores_path = os.path.join(preprocessed_dir, "flores_plus.jsonl")
if os.path.exists(flores_path):
    lang_counts = {lang: 0 for lang in OLD_LANGUAGES}
    
    with open(flores_path, 'r', encoding='utf-8') as f:
        for line in f:
            record = json.loads(line)
            label = record.get('label')
            if label in lang_counts and lang_counts[label] < MAX_SAMPLES_PER_OLD_LANG:
                old_lang_samples.append(record)
                lang_counts[label] += 1
                
    print(f"Sampled old language distributions from flores_plus:")
    for lang, count in lang_counts.items():
        print(f"  {lang}: {count}")
else:
    print(f"WARNING: {flores_path} not found. Cannot create a robust mixed validation set.")

# Combine with val.jsonl
mixed_records = []
val_jsonl_path = os.path.join(output_dir, "val.jsonl")
if os.path.exists(val_jsonl_path):
    with open(val_jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            mixed_records.append(json.loads(line))

mixed_records.extend(old_lang_samples)
random.shuffle(mixed_records) # Shuffle to distribute classes evenly

mixed_path = os.path.join(output_dir, "val_mixed.jsonl")
with open(mixed_path, 'w', encoding='utf-8') as f:
    for record in mixed_records:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

print(f"\nCreated {mixed_path} with {len(mixed_records)} total mixed records.")



Creating mixed validation set (val_mixed.jsonl) to track Catastrophic Forgetting...
Sampled old language distributions from flores_plus:
  eng: 500
  tam: 500
  hin: 500
  ben: 500
  arb: 500
  fra: 500
  deu: 500

Created datasets/finetuning\val_mixed.jsonl with 10486 total mixed records.
